In [9]:
import kagglehub

# Download latest version
path = kagglehub.competition_download('playground-series-s6e1')

print("Path to competition files:", path)

Path to competition files: /kaggle/input/competitions/playground-series-s6e1


In [10]:
import numpy as np 
import pandas as pd

In [32]:
dataset = pd.read_csv('/kaggle/input/competitions/playground-series-s6e1/train.csv')
test = pd.read_csv('/kaggle/input/competitions/playground-series-s6e1/test.csv')


In [33]:
dataset


,id,age,gender,course,study_hours,class_attendance,internet_access,sleep_hours,sleep_quality,study_method,facility_rating,exam_difficulty,exam_score
0,0,21,female,b.sc,7.91,98.8,no,4.9,average,online videos,low,easy,78.300
1,1,18,other,diploma,4.95,94.8,yes,4.7,poor,self-study,medium,moderate,46.700
2,2,20,female,b.sc,4.68,92.6,yes,5.8,poor,coaching,high,moderate,99.000
3,3,19,male,b.sc,2.00,49.5,yes,8.3,average,group study,high,moderate,63.900
4,4,23,male,bca,7.65,86.9,yes,9.6,good,self-study,high,easy,100.000
...,...,...,...,...,...,...,...,...,...,...,...,...,...
629995,629995,18,female,b.tech,4.86,70.7,yes,4.1,good,mixed,high,moderate,69.500
629996,629996,21,female,ba,7.08,54.4,yes,4.5,average,mixed,low,moderate,78.900
629997,629997,24,male,bca,0.64,44.2,yes,4.3,poor,online videos,low,moderate,19.599
629998,629998,20,male,b.com,1.54,75.1,yes,8.2,average,group study,high,moderate,59.100


In [34]:
test

,id,age,gender,course,study_hours,class_attendance,internet_access,sleep_hours,sleep_quality,study_method,facility_rating,exam_difficulty
0,630000,24,other,ba,6.85,65.2,yes,5.2,poor,group study,high,easy
1,630001,18,male,diploma,6.61,45.0,no,9.3,poor,coaching,low,easy
2,630002,24,female,b.tech,6.60,98.5,yes,6.2,good,group study,medium,moderate
3,630003,24,male,diploma,3.03,66.3,yes,5.7,average,mixed,medium,moderate
4,630004,20,female,b.tech,2.03,42.4,yes,9.2,average,coaching,low,moderate
...,...,...,...,...,...,...,...,...,...,...,...,...
269995,899995,21,other,b.com,2.55,82.3,yes,8.4,average,mixed,medium,hard
269996,899996,17,female,b.com,0.49,46.4,yes,8.8,good,mixed,low,easy
269997,899997,22,male,bba,6.62,74.7,yes,5.5,good,coaching,high,easy
269998,899998,22,other,ba,4.08,51.8,yes,8.7,poor,online videos,high,moderate


In [15]:
target = "exam_score"
feature = [col for col in dataset.columns if col not in ["id", target]]


In [21]:
ordinal_maps = {
    "sleep_quality": {
        "poor": 0,
        "average": 1,
        "good": 2},
    "facility_rating":{
        "low": 0,
        "medium": 1,
        "good": 2
    },
    "exam_difficulty":{
        "easy": 0,
        "moderate": 1,
        "hard": 2
    },
    "internet_access":{
        "no": 0,
        "yes": 1
    },
    
}

In [54]:
for col, mapping in ordinal_maps.items():
    dataset[col] = dataset[col].map(mapping)
    test[col] = test[col].map(mapping)
  
    

In [55]:
cat_cols = ["gender", "course", "study_method"]

In [56]:
for col in cat_cols:
    dataset[col] = dataset[col].astype("category")

In [65]:
X_test = test[feature].copy()

for col in cat_cols:
    X_test[col] = X_test[col].astype(X_train[col].dtype)

In [58]:
X_train = dataset[feature]

In [59]:
Y_train = dataset[target]

In [60]:
from lightgbm import LGBMRegressor



In [61]:
model = LGBMRegressor(
    objective="regression", metric="rmse",random_state=42
)

In [66]:
model.fit(X_train, Y_train)

[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.005002 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 587
[LightGBM] [Info] Number of data points in the train set: 630000, number of used features: 7
[LightGBM] [Info] Start training from score 62.506672


LGBMRegressor(metric='rmse', objective='regression', random_state=42)

In [67]:
Y_pred = model.predict(X_test)

In [68]:
submission = pd.DataFrame(
    {
        "id": test["id"], "exam_score": Y_pred
    }
)

In [69]:
submission.to_csv("submission.csv", index = False)